In [1]:
from google.colab import drive
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to your CSV file
file_path = '/content/drive/MyDrive/Colab/world_cup_match_data.csv'

df = pd.read_csv(file_path)
print("CSV imported successfully!")
print(df.head())


Mounted at /content/drive
CSV imported successfully!
    timestamp              date_GMT    status  attendance home_team_name  \
0  1781204400  Jun 11 2026 - 7:00pm  complete       80824         Mexico   
1  1781229600  Jun 12 2026 - 2:00am  complete       44985    South Korea   
2  1781290800  Jun 12 2026 - 7:00pm  complete       43002         Canada   
3  1781312400  Jun 13 2026 - 1:00am  complete       70492          USMNT   
4  1781377200  Jun 13 2026 - 7:00pm  complete       67966          Qatar   

           away_team_name  referee  Game Week  Pre-Match PPG (Home)  \
0            South Africa      NaN        1.0                   0.0   
1          Czech Republic      NaN        1.0                   0.0   
2  Bosnia and Herzegovina      NaN        1.0                   0.0   
3                Paraguay      NaN        1.0                   0.0   
4             Switzerland      NaN        1.0                   0.0   

   Pre-Match PPG (Away)  ...  odds_ft_home_team_win  odds_ft_dr

In [2]:
df = df[
        [
            'home_team_name',
            'away_team_name',
            'home_team_goal_count',
            'away_team_goal_count',
            'home_team_possession',
            'away_team_possession'
        ]
    ].copy()

# Determine winning team
# 0 = Home team wins
# 1 = Away team wins
# -1 = Draw
df['winning_team'] = df.apply(
    lambda row: 0 if row['home_team_goal_count'] > row['away_team_goal_count']
    else 1 if row['away_team_goal_count'] > row['home_team_goal_count']
    else -1,
    axis=1
)
df = df[df['winning_team'] != -1].copy()

df['possession_difference'] = df.apply(
    lambda row: (
        row['home_team_possession'] - row['away_team_possession']
        if row['winning_team'] == 0
        else row['away_team_possession'] - row['home_team_possession']
    ),
    axis=1
)

df.reset_index(drop=True, inplace=True)
print(df.head())

  home_team_name  away_team_name  home_team_goal_count  away_team_goal_count  \
0         Mexico    South Africa                     2                     0   
1    South Korea  Czech Republic                     2                     1   
2          USMNT        Paraguay                     4                     1   
3          Haiti        Scotland                     0                     1   
4      Australia          Turkey                     2                     0   

   home_team_possession  away_team_possession  winning_team  \
0                    61                    39             0   
1                    62                    38             0   
2                    65                    35             0   
3                    54                    46             1   
4                    28                    72             0   

   possession_difference  
0                     22  
1                     24  
2                     30  
3                     -8  
4    

In [3]:
import scipy.stats as st
import numpy as np

# x_bar = st.tmean(df['possession_difference'])
# s = st.tstd(df['possession_difference'])
# print("x_bar:", x_bar)
# print("s:", s)

#H0: true mean difference = 0 -> no difference between the possesion between
#winning team and losing team
#H1: true mean > 0 -> winning team has more possesion than the losing team

t_stat, p_value = st.ttest_1samp(df['possession_difference'],0, alternative='greater')

print("t-statistic:", t_stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("\t We reject the null hypothesis.")
else:
    print("\t We fail to reject the null hypothesis.")

t-statistic: 5.09386411450459
p-value: 1.1667953373401071e-06
	 We reject the null hypothesis.


In [4]:
df.describe(include='all')

,home_team_name,away_team_name,home_team_goal_count,away_team_goal_count,home_team_possession,away_team_possession,winning_team,possession_difference
count,80,80,80.000000,80.000000,80.000000,80.000000,80.000000,80.000000
unique,41,44,NaN,NaN,NaN,NaN,NaN,NaN
top,France,England,NaN,NaN,NaN,NaN,NaN,NaN
freq,6,4,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,1.975000,1.350000,51.325000,48.675000,0.375000,12.400000
std,NaN,NaN,1.574922,1.322636,12.476586,12.476586,0.487177,21.773053
min,NaN,NaN,0.000000,0.000000,24.000000,21.000000,0.000000,-58.000000
25%,NaN,NaN,1.000000,0.000000,42.750000,38.750000,0.000000,1.000000
50%,NaN,NaN,2.000000,1.000000,52.000000,48.000000,0.000000,15.000000
75%,NaN,NaN,3.000000,2.000000,61.250000,57.250000,1.000000,28.000000


In [5]:
df2 = pd.read_csv(file_path)
df2 = df2[
        [
            'home_team_name',
            'away_team_name',
            'home_team_shots',
            'away_team_shots',
            'home_team_possession',
            'away_team_possession'
        ]
    ].copy()
df2['match'] = (
    df2['home_team_name'] + ' vs ' + df2['away_team_name']
)

df_long = pd.concat([
    df2[['match','home_team_name', 'home_team_possession', 'home_team_shots']]
        .rename(columns={
            'home_team_name': 'team',
            'home_team_possession': 'possession',
            'home_team_shots': 'shots'
        }),

    df2[['match','away_team_name', 'away_team_possession', 'away_team_shots']]
        .rename(columns={
            'away_team_name': 'team',
            'away_team_possession': 'possession',
            'away_team_shots': 'shots'
        })
], ignore_index=True)

df_long.head()

,match,team,possession,shots
0,Mexico vs South Africa,Mexico,61,16
1,South Korea vs Czech Republic,South Korea,62,15
2,Canada vs Bosnia and Herzegovina,Canada,61,13
3,USMNT vs Paraguay,USMNT,65,16
4,Qatar vs Switzerland,Qatar,32,6


In [6]:
import scipy.stats as st

# Split teams into two groups
#high_possesion > 50% possesion
#low_possesion < 50% possesion
high_possession = df_long[df_long['possession'] > 50]['shots']
low_possession = df_long[df_long['possession'] < 50]['shots']

# Two-sample t-test
#H0: high-possesion mean shot = low possesion mean shot
#H1: high possesion mean shot > low possesion mean shot
t_stat, p_value = st.ttest_ind(
    high_possession,
    low_possession,
    alternative='greater'
)

print("High possession mean:", high_possession.mean())
print("Low possession mean:", low_possession.mean())
print("t-statistic:", t_stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Reject the null hypothesis.")
else:
    print("Fail to reject the null hypothesis.")

High possession mean: 15.509615384615385
Low possession mean: 9.259615384615385
t-statistic: 8.144606733516998
p-value: 1.8055111538697105e-14
Reject the null hypothesis.


In [7]:
df3 = pd.read_csv(file_path)
df3 = df3[
        [
            'home_team_name',
            'away_team_name',
            'home_team_goal_count',
            'away_team_goal_count',
            'home_team_yellow_cards',
            'away_team_yellow_cards',
            'home_team_red_cards',
            'away_team_red_cards'
        ]
    ].copy()

# Determine winning team
# 0 = Home team wins
# 1 = Away team wins
# -1 = Draw
df3['winning_team'] = df3.apply(
    lambda row: 0 if row['home_team_goal_count'] > row['away_team_goal_count']
    else 1 if row['away_team_goal_count'] > row['home_team_goal_count']
    else -1,
    axis=1
)
df3 = df3[df3['winning_team'] != -1].copy()
df3['home_team_total_card']=df3['home_team_yellow_cards']+df3['home_team_red_cards']
df3['away_team_total_card']=df3['away_team_yellow_cards']+df3['away_team_red_cards']
df3['match'] = (
    df3['home_team_name'] + ' vs ' + df3['away_team_name']
)

df3_long = pd.concat([
    df3[['match', 'home_team_name', 'home_team_total_card', 'winning_team']]
        .rename(columns={
            'home_team_name': 'team',
            'home_team_total_card': 'total_card'
        })
        .assign(win=lambda x: (x['winning_team'] == 0).astype(int)),

    df3[['match', 'away_team_name', 'away_team_total_card', 'winning_team']]
        .rename(columns={
            'away_team_name': 'team',
            'away_team_total_card': 'total_card'
        })
        .assign(win=lambda x: (x['winning_team'] == 1).astype(int))
], ignore_index=True)

df3_long = df3_long.drop(columns='winning_team')
df3_long

,match,team,total_card,win
0,Mexico vs South Africa,Mexico,2,1
1,South Korea vs Czech Republic,South Korea,1,1
2,USMNT vs Paraguay,USMNT,1,1
3,Haiti vs Scotland,Haiti,1,0
4,Australia vs Turkey,Australia,0,1
...,...,...,...,...
155,Argentina vs Switzerland,Switzerland,2,0
156,France vs Spain,Spain,1,1
157,England vs Argentina,Argentina,3,1
158,France vs England,England,0,1


In [12]:
winning_team = df3_long[df3_long['win']==1]['total_card']
losing_team = df3_long[df3_long['win']==0]['total_card']

# two sample t_test
# H0: Total card winning team = total card losing team
# H1: Total card losing team > wining team -> making more foul can lead to losing
t_stat, p_value = st.ttest_ind(
    losing_team,
    winning_team,
    alternative='greater'
)

print("Losing team total card mean:", losing_team.mean())
print("Winning team total card mean:", winning_team.mean())
print("t-statistic:", t_stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Reject the null hypothesis.")
else:
    print("Fail to reject the null hypothesis.")


Losing team total card mean: 1.5875
Winning team total card mean: 1.0375
t-statistic: 2.856658037741833
p-value: 0.0024287960427460138
Reject the null hypothesis.
